# AI데이터사이언스 13강 — 평균 예측 (Estimating the Mean)

**2026-1학기 · ICT융합학부 컴퓨터공학트랙 · 김대환**

이 노트북은 강의자료 `AI데이터사이언스13.pdf` 의 흐름을 그대로 따라가는 **실습 파일**입니다.
원본 강의는 UC Berkeley *Data 8* 의 `datascience` 라이브러리를 사용하지만,
이 실습은 어디서나 바로 실행되도록 **numpy / pandas / matplotlib / scipy** 표준 라이브러리로 재구성했습니다.

## 목차
1. 개요
2. 평균의 성질
3. 변동성 (Variability)
4. SD와 정규곡선 (SD and Normal Curve)
5. 중심극한정리
6. 표본평균의 변동성
7. 표본크기 결정

<div align="center">
<img src="images/page_01.png" width="760" alt="강의 슬라이드 p.1"><br>
<sub>📑 강의 슬라이드 p.1</sub>
</div>

## 0. 준비 — 라이브러리 & 헬퍼 함수

아래 셀을 가장 먼저 실행하세요.

데이터셋(`san_francisco_2015.csv`, `nba2013.csv`, `baby.csv`, `united_summer2015.csv`)은 **이 노트북과 같은 `13장` 폴더에 이미 들어 있습니다.** `load_data()` 는 ① 같은 폴더 → ② 현재 작업 폴더 → ③ (없을 때만) GitHub 다운로드 순으로 찾으므로 **인터넷 없이도 실행**됩니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True
np.random.seed(13)  # 재현성

# 이 노트북과 같은 폴더(13장)에 csv 파일들이 함께 들어 있습니다.
try:
    DATA_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    DATA_DIR = os.getcwd()  # Jupyter/Colab 환경: 현재 작업 폴더
BASE_URL = 'https://raw.githubusercontent.com/data-8/materials-sp18/master/lec/'

def load_data(filename):
    """Data 8 강의 데이터셋 로드 (로컬 폴더 -> 원격 URL 순으로 시도)."""
    local_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(local_path):              # ① 같은 폴더의 csv 우선 (오프라인 OK)
        return pd.read_csv(local_path)
    if os.path.exists(filename):                # ② 현재 작업 디렉터리
        return pd.read_csv(filename)
    try:
        return pd.read_csv(BASE_URL + filename)  # ③ 둘 다 없으면 GitHub에서 다운로드
    except Exception as e:
        raise RuntimeError(f'{filename} 를 불러올 수 없습니다: {e}')

def percentile(p, arr):
    """Data 8 스타일 백분위수 (p 번째 percentile)."""
    arr = np.sort(np.asarray(arr, dtype=float))
    n = len(arr)
    rank = int(np.ceil(p / 100 * n))
    return arr[max(rank, 1) - 1]

def standard_units(arr):
    """표준 단위(z = (값 - 평균) / SD)로 변환."""
    arr = np.asarray(arr, dtype=float)
    return (arr - np.mean(arr)) / np.std(arr)

def hist_percent(data, bins, xlabel='', title='', fulcrum=None, ax=None):
    """y축이 'Percent per unit' 인 밀도 히스토그램 (Data 8 의 .hist 와 동일 스케일)."""
    if ax is None:
        _, ax = plt.subplots()
    data = np.asarray(data, dtype=float)
    # density=True : 면적의 합 = 1 인 밀도 히스토그램 (단위 폭당 비율)
    ax.hist(data, bins=bins, density=True, edgecolor='white', color='#3b5d78')
    ax.set_ylabel('Percent per unit')
    ax.set_xlabel(xlabel)
    if title:
        ax.set_title(title)
    if fulcrum is not None:
        ax.plot(fulcrum, 0, marker='^', color='red', markersize=12, clip_on=False)
    return ax

print('준비 완료 ✔  (데이터 폴더:', DATA_DIR, ')')

> 참고: `hist_percent` 는 `density=True` 를 사용해 면적의 합이 1이 되는 밀도 히스토그램을 그립니다. 원본 강의자료의 y축 라벨 *Percent per unit* 과 동일한 의미(단위 폭당 비율)입니다.

---
# 1. 개요

- 표본평균의 **경험적 분포**는 모집단의 분포와 무관하게 항상 **종 모양(bell-shaped)** 이다.

### 핵심 질문
- 평균은 정확히 무엇을 측정하는가?
- 대부분의 데이터는 평균에서 얼마나 떨어져 있는가?
- 표본 크기는 표본평균의 변동성과 어떤 관계인가?
- 왜 랜덤 표본평균의 경험적 분포는 종 모양이 되는가?
- 표본평균을 추론에 어떻게 효과적으로 사용할 수 있는가?

<div align="center">
<img src="images/page_04.png" width="760" alt="강의 슬라이드 p.4"><br>
<sub>📑 강의 슬라이드 p.4</sub>
</div>

---
# 2. 평균의 성질

## 2.1 평균 (Mean / Average)

$$\text{평균} = \frac{\text{모든 원소의 합}}{\text{원소의 개수}}$$

**평균의 기본 성질**
- 컬렉션의 원소일 필요가 없음
- 원소가 모두 정수여도 반드시 정수는 아님
- 최솟값과 최댓값 사이에 위치함
- 두 극단의 중간점일 필요가 없다 → 절반의 원소가 평균 이상이라는 보장 없음
- 원래 데이터와 동일한 단위를 가짐

<div align="center">
<img src="images/page_06.png" width="760" alt="강의 슬라이드 p.6"><br>
<sub>📑 강의 슬라이드 p.6</sub>
</div>

In [ ]:
not_symmetric = np.array([2, 3, 3, 9])
print('np.average :', np.average(not_symmetric))
print('np.mean    :', np.mean(not_symmetric))

## 2.2 평균은 히스토그램의 무게중심 (지렛점, fulcrum)

평균은 원소 **개수**가 아니라 **분포(값의 비율)** 에만 의존한다.
→ 같은 분포를 가진 두 컬렉션은 반드시 같은 평균을 가진다.

$$\text{mean} = \frac{2+3+3+9}{4} = 2\cdot\tfrac14 + 3\cdot\tfrac24 + 9\cdot\tfrac14 = 4.25$$

<div align="center">
<img src="images/page_07.png" width="760" alt="강의 슬라이드 p.7"><br>
<sub>📑 강의 슬라이드 p.7</sub>
</div>

In [ ]:
same_distribution = np.array([2, 2, 3, 3, 3, 3, 9, 9])  # 같은 분포, 원소 수만 2배
print('not_symmetric   평균:', np.mean(not_symmetric))
print('same_distribution 평균:', np.mean(same_distribution))

ax = hist_percent(not_symmetric, bins=np.arange(1.5, 10.5, 1),
                  xlabel='not symmetric', fulcrum=np.mean(not_symmetric))
ax.set_title('평균 = 무게중심(지렛점) = 4.25')
plt.show()

## 2.3 평균 vs 중앙값(Median) — 왜도(Skewness)의 영향

히스토그램이 한쪽으로 치우치면, **평균은 중앙값에서 꼬리 방향으로 끌려간다.**

| | 대칭분포 | 우편향 (Right-skewed) |
|---|---|---|
| 예시 | `[2, 3, 3, 4]` | `[2, 3, 3, 9]` |
| 평균 | 3 | 4.25 |
| 중앙값 | 3 | 3 |
| 형태 | 평균에 대해 대칭 | 평균이 꼬리 방향으로 당겨짐 |

<div align="center">
<img src="images/page_08.png" width="760" alt="강의 슬라이드 p.8"><br>
<sub>📑 강의 슬라이드 p.8</sub>
</div>

In [ ]:
symmetric = np.array([2, 3, 3, 4])
for name, arr in [('symmetric', symmetric), ('not_symmetric', not_symmetric)]:
    print(f'{name:14s}  평균={np.mean(arr):.2f}  중앙값={np.median(arr):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hist_percent(symmetric, bins=np.arange(1.5, 11, 1), xlabel='symmetric',
             fulcrum=np.mean(symmetric), ax=axes[0])
hist_percent(not_symmetric, bins=np.arange(1.5, 11, 1), xlabel='not symmetric',
             fulcrum=np.mean(not_symmetric), ax=axes[1])
axes[0].set_title('대칭: 평균 = 중앙값')
axes[1].set_title('우편향: 평균 > 중앙값')
plt.tight_layout(); plt.show()

## 2.4 예제 — 샌프란시스코 2015 공무원 급여

오른쪽으로 길게 치우친(우편향) 실제 데이터에서 평균 > 중앙값 임을 확인한다.

<div align="center">
<img src="images/page_09.png" width="760" alt="강의 슬라이드 p.9"><br>
<sub>📑 강의 슬라이드 p.9</sub>
</div>

In [ ]:
sf = load_data('san_francisco_2015.csv')
sf2015 = sf[sf['Total Compensation'] > 10000]

hist_percent(sf2015['Total Compensation'], bins=np.arange(10000, 700000, 25000),
             xlabel='Total Compensation', title='SF 공무원 급여 (우편향)')
plt.show()

compensation = sf2015['Total Compensation'].values
print('중앙값 (percentile 50):', percentile(50, compensation))
print('평균                  :', np.mean(compensation))
print('→ 우편향이므로 평균 > 중앙값')

---
# 3. 변동성 (Variability)

## 3.1 표준편차(Standard Deviation) 계산 — 단계별

1. 평균 계산
2. 편차(deviation) = 각 값 − 평균
3. 편차를 제곱 (squared deviation)
4. 분산(variance) = 제곱 편차의 평균
5. 표준편차(SD) = √분산

$$\text{SD} = \sqrt{\text{제곱 편차의 평균}}$$

<div align="center">
<img src="images/page_11.png" width="760" alt="강의 슬라이드 p.11"><br>
<sub>📑 강의 슬라이드 p.11</sub>
</div>

In [ ]:
any_numbers = np.array([1, 2, 2, 10])

# 1) 평균
mean = np.mean(any_numbers)
# 2) 편차
deviations = any_numbers - mean
# 3) 제곱 편차
sq_dev = deviations ** 2
# 4) 분산
variance = np.mean(sq_dev)
# 5) 표준편차
sd = variance ** 0.5

steps = pd.DataFrame({'Value': any_numbers,
                      'Deviation from Average': deviations,
                      'Squared Deviations': sq_dev})
print(steps.to_string(index=False))
print()
print('평균    :', mean)
print('분산    :', variance)
print('SD(직접):', sd)
print('np.std  :', np.std(any_numbers))  # 동일해야 함

## 3.2 예시 — NBA 선수 데이터

가장 키 크고 가장 작은 선수도 평균에서 불과 3 SD 이내에 있다.
→ **SD는 데이터가 얼마나 흩어져 있는지를 요약하는 유용한 척도**이다.

<div align="center">
<img src="images/page_12.png" width="760" alt="강의 슬라이드 p.12"><br>
<sub>📑 강의 슬라이드 p.12</sub>
</div>

In [ ]:
nba = load_data('nba2013.csv')
heights = nba['Height'].values

mean_height = np.mean(heights)
sd_height = np.std(heights)
print('평균 키:', round(mean_height, 2), 'inch')
print('SD     :', round(sd_height, 2), 'inch')

hist_percent(heights, bins=np.arange(68, 88, 1), xlabel='Height',
             title='NBA 선수 키 분포')
plt.show()

In [ ]:
# 가장 큰 / 가장 작은 선수가 평균에서 몇 SD 떨어져 있나? (표준 단위)
tallest = heights.max()
shortest = heights.min()
print('가장 큰 선수 :', tallest, 'inch ->', round((tallest - mean_height) / sd_height, 2), 'SD')
print('가장 작은 선수:', shortest, 'inch ->', round((shortest - mean_height) / sd_height, 2), 'SD')
print('→ 모두 평균에서 3 SD 이내')

## 3.3 체비쇼프 부등식 (Chebyshev's Bounds)

체비쇼프는 **하한(lower bound)** 을 제공할 뿐, 정확한 값이나 근사치가 아니다. 하지만 **어떤 분포에도 보편적으로 적용**된다.

> 모든 수치 데이터 리스트에 대해, **"평균 ± z SD"** 범위에 속하는 원소의 비율은 최소 $1 - 1/z^2$ ($z>1$)

| 범위 | 최소 비율 |
|---|---|
| 평균 ± 2 SD | $1 - 1/4 = 0.75$ |
| 평균 ± 3 SD | $1 - 1/9 \approx 0.89$ |
| 평균 ± 4.5 SD | $1 - 1/4.5^2 \approx 0.95$ |

<div align="center">
<img src="images/page_13.png" width="760" alt="강의 슬라이드 p.13"><br>
<sub>📑 강의 슬라이드 p.13</sub>
</div>

In [ ]:
def chebyshev_lower_bound(z):
    return 1 - 1 / z**2

def proportion_within(data, z):
    """실제로 평균 ± z SD 안에 들어온 데이터 비율."""
    m, s = np.mean(data), np.std(data)
    inside = (data >= m - z*s) & (data <= m + z*s)
    return np.mean(inside)

ages = nba['Age in 2013'].values
for z in [2, 3, 4.5]:
    print(f'z={z}: 체비쇼프 하한={chebyshev_lower_bound(z):.3f},'
          f'  NBA 나이 실제 비율={proportion_within(ages, z):.3f}')

## 3.4 표준 단위 (Standard Units)

$$z = \frac{\text{value} - \text{average}}{\text{SD}}$$

체비쇼프 부등식에 의해, 표준 단위 z 값은 대부분 (−5, 5) 범위에 머문다.

**예시 — United Airlines 결항/지연 데이터**

<div align="center">
<img src="images/page_14.png" width="760" alt="강의 슬라이드 p.14"><br>
<sub>📑 강의 슬라이드 p.14</sub>
</div>

In [ ]:
united = load_data('united_summer2015.csv')
delay = united['Delay'].values
united = united.assign(**{'Delay (Standard Units)': standard_units(delay)})

# 표준 단위가 (-3, 3) 안에 들어오는 비율
su = united['Delay (Standard Units)'].values
within_3 = np.mean((su > -3) & (su < 3))
print('|z| < 3 인 비율:', round(within_3, 4))

print('\n가장 지연이 큰 항공편 (표준 단위 상위):')
print(united.sort_values('Delay', ascending=False)[['Flight Number', 'Destination',
      'Delay', 'Delay (Standard Units)']].head().to_string(index=False))

hist_percent(su, bins=np.arange(-5, 15.5, 0.5), xlabel='Delay (Standard Units)',
             title='지연시간의 표준 단위 분포')
plt.show()

---
# 4. SD와 정규곡선 (SD and Normal Curve)

## 4.1 종 모양 분포에서의 SD 찾기 — 산모 신장 데이터

- 종 모양 분포에서 **SD = 평균과 변곡점(inflection point) 사이의 거리**
- 변곡점: 곡선이 "위로 오목(upside-down cup)"에서 "아래로 오목(right-way-up cup)"으로 바뀌는 지점
- 즉, z = ±1 지점이 변곡점 → 평균 ± 1 SD

<div align="center">
<img src="images/page_16.png" width="760" alt="강의 슬라이드 p.16"><br>
<sub>📑 강의 슬라이드 p.16</sub>
</div>

In [ ]:
baby = load_data('baby.csv')
m_heights = baby['Maternal Height'].values

mean_h = round(np.mean(m_heights), 1)
sd_h = round(np.std(m_heights), 1)
print('평균 :', mean_h, 'inch')
print('SD   :', sd_h, 'inch')
print(f'변곡점 ≈ 평균 ± 1 SD = {mean_h} ± {sd_h} = ({mean_h-sd_h}, {mean_h+sd_h})')

ax = hist_percent(m_heights, bins=np.arange(55.5, 72.5, 1),
                  xlabel='Maternal Height (inch)', title='산모 신장 (종 모양)')
for x in [mean_h - sd_h, mean_h + sd_h]:
    ax.axvline(x, color='red', linestyle='--')
ax.axvline(mean_h, color='green', linestyle='-')
plt.show()

## 4.2 표준 정규 곡선 (Standard Normal Curve)

모든 종 모양 히스토그램은 축의 레이블만 다를 뿐 본질적으로 동일한 하나의 곡선이다.
표준 단위로 변환한 이 기본 곡선을 **표준 정규곡선**이라 한다.

$$\phi(z) = \frac{1}{\sqrt{2\pi}} e^{-\frac12 z^2}, \quad -\infty < z < \infty$$

**주요 성질**
- 곡선 아래 전체 넓이 = 1
- z = 0 에 대해 대칭 → 평균 = 중앙값 = 0
- 변곡점 위치: z = −1, z = +1
- SD = 1

<div align="center">
<img src="images/page_17.png" width="760" alt="강의 슬라이드 p.17"><br>
<sub>📑 강의 슬라이드 p.17</sub>
</div>

In [ ]:
z = np.linspace(-3.5, 3.5, 400)
phi = stats.norm.pdf(z)  # = (1/sqrt(2*pi)) * exp(-z**2/2)

plt.plot(z, phi, color='black')
plt.fill_between(z, phi, alpha=0.1)
plt.title('Normal Curve  ~  (μ=0, σ=1)')
plt.xlabel('z'); plt.ylabel('φ(z)')
plt.show()

## 4.3 표준 정규 CDF 와 68–95–99.7 법칙

정규곡선은 일반적인 미적분으로 적분이 불가능하여, Python 의 `scipy.stats.norm.cdf` 를 사용한다.

| 범위 | 모든 분포 (체비쇼프 하한) | 정규분포 근사 |
|---|---|---|
| 평균 ± 1 SD | at least 0% | about 68% |
| 평균 ± 2 SD | at least 75% | about 95% |
| 평균 ± 3 SD | at least 88.888% | about 99.73% |

→ 분포가 정규분포임을 알면 하한 대신 정확한 근사치를 사용할 수 있어 훨씬 강력하다.

<div align="center">
<img src="images/page_18.png" width="760" alt="강의 슬라이드 p.18"><br>
<sub>📑 강의 슬라이드 p.18</sub>
</div>

In [ ]:
# P(-1 < Z < 1) 등을 cdf 로 계산
for zval, pct in [(1, '68%'), (2, '95%'), (3, '99.7%')]:
    area = stats.norm.cdf(zval) - stats.norm.cdf(-zval)
    print(f'평균 ± {zval} SD : {area:.4f}  (약 {pct})')

print()
print('stats.norm.cdf(1)              =', stats.norm.cdf(1))
print('1 - stats.norm.cdf(1)          =', 1 - stats.norm.cdf(1))
print('cdf(1) - cdf(-1)               =', stats.norm.cdf(1) - stats.norm.cdf(-1))
print('cdf(2) - cdf(-2)               =', stats.norm.cdf(2) - stats.norm.cdf(-2))

---
# 5. 중심극한정리 (Central Limit Theorem)

> **(정의)** 대형 랜덤 표본(복원 추출)의 합 또는 평균의 확률 분포는, 모집단의 분포에 무관하게, 근사적으로 정규분포를 따른다.

모집단 분포를 거의 모르는 상황에서도 대형 랜덤 표본만 있으면 추론이 가능하다.

## 5.1 예시 1 — 룰렛 순이익

이항분포(빨강에 베팅)임에도 불구하고, 큰 표본의 합/평균의 분포는 종 모양이 된다.

<div align="center">
<img src="images/page_20.png" width="760" alt="강의 슬라이드 p.20"><br>
<sub>📑 강의 슬라이드 p.20</sub>
</div>

In [ ]:
# 룰렛 휠: 38칸 (빨강 18, 검정 18, 초록 2). '빨강'에 베팅: 빨강이면 +1, 아니면 -1
def red_winnings(color):
    return 1 if color == 'red' else -1

colors = np.array(['red'] * 18 + ['black'] * 18 + ['green'] * 2)
winnings_red = np.array([red_winnings(c) for c in colors])

num_bets = 400
repetitions = 10000
net_gain_red = np.array([
    np.random.choice(winnings_red, num_bets).sum()
    for _ in range(repetitions)
])

hist_percent(net_gain_red, bins=np.arange(-80, 50, 6), xlabel='Net Gain on Red',
             title='룰렛: 400회 베팅 순이익 (10000번 반복)')
plt.show()

average_per_bet = 1 * (18/38) + (-1) * (20/38)
print('베팅당 기대 순이익 :', round(average_per_bet, 6))
print('400회 기대 순이익  :', round(400 * average_per_bet, 4))
print('시뮬레이션 평균    :', round(np.mean(net_gain_red), 4))
print('시뮬레이션 SD      :', round(np.std(net_gain_red), 4))

## 5.2 예시 2 — 비행 지연 평균

모집단(13,825개 국내 비행 지연)의 분포가 **우편향**임에도 불구하고,
큰 표본(n=400)의 평균의 분포는 **종 모양**이 된다.

<div align="center">
<img src="images/page_21.png" width="760" alt="강의 슬라이드 p.21"><br>
<sub>📑 강의 슬라이드 p.21</sub>
</div>

In [ ]:
# 모집단 분포: 우편향
hist_percent(delay, bins=np.arange(-20, 300, 10), xlabel='Delay',
             title='모집단: 비행 지연 (우편향)')
plt.show()
print('모집단 평균:', round(np.mean(delay), 4))
print('모집단 SD  :', round(np.std(delay), 4))

In [ ]:
sample_size = 400
repetitions = 10000
means = np.array([
    np.mean(np.random.choice(delay, sample_size))
    for _ in range(repetitions)
])

hist_percent(means, bins=np.arange(10, 25, 0.5), xlabel='Sample Mean',
             title=f'표본평균(n={sample_size})의 분포 → 종 모양')
plt.show()
print('표본평균들의 평균:', round(np.mean(means), 4))
print('표본평균들의 SD  :', round(np.std(means), 4))

## 5.3 예시 3 — 멘델 완두콩

표본 비율(sample proportion)도 결국 0/1 값의 평균이므로 CLT 가 적용된다.
**표본 크기가 클수록 표본 비율의 분포 변동성이 감소**한다.

<div align="center">
<img src="images/page_22.png" width="760" alt="강의 슬라이드 p.22"><br>
<sub>📑 강의 슬라이드 p.22</sub>
</div>

In [ ]:
# 멘델: 보라:흰 = 3:1 → 보라 비율 0.75
model = np.array(['Purple', 'Purple', 'Purple', 'White'])

def sample_proportion_purple(n):
    sample = np.random.choice(model, n)
    return np.mean(sample == 'Purple')

reps = 10000
props_200 = np.array([sample_proportion_purple(200) for _ in range(reps)])
props_800 = np.array([sample_proportion_purple(800) for _ in range(reps)])

bins = np.arange(0.65, 0.85, 0.01)
plt.hist(props_200, bins=bins, density=True, alpha=0.6, label='n = 200',
         color='#3b5d78', edgecolor='white')
plt.hist(props_800, bins=bins, density=True, alpha=0.6, label='n = 800',
         color='#d9a441', edgecolor='white')
plt.xlabel('Sample Proportion of Purple'); plt.ylabel('Percent per unit')
plt.title('표본 크기가 클수록 변동성 감소'); plt.legend()
plt.show()
print('n=200  표본비율 SD:', round(np.std(props_200), 4))
print('n=800  표본비율 SD:', round(np.std(props_800), 4))

---
# 6. 표본평균의 변동성

## 6.1 표본평균도 확률변수

- 모집단 전체를 조사하는 것은 현실적으로 불가능 → 랜덤 표본의 표본평균 $\bar{x}$ 로 모집단 평균 $\mu$ 를 추정
- 표본을 새로 뽑을 때마다 $\bar{x}$ 값이 달라짐 → $\bar{x}$ 는 **확률변수**

> **표본평균은 모집단 평균의 불편 추정량(unbiased estimate)**
> → 모든 가능한 표본에 대한 평균들의 평균 = 모집단 평균

| $\bar{x}$ 는 어떤 분포를 따르는가? | $\bar{x}$ 는 μ 주변에서 얼마나 퍼져 있는가? |
|---|---|
| 중심극한정리(CLT)를 따름 | 표본 크기가 달라지면 종 모양의 **너비**가 달라짐 |
| 표본이 충분히 크면 ≈ 정규분포 | 이 변동성을 정확히 **수치로** 표현 필요 |
| 종 모양의 중심 = 모집단 평균 μ | 표본 크기 n 과 변동성은 어떤 관계? |

<div align="center">
<img src="images/page_24.png" width="760" alt="강의 슬라이드 p.24"><br>
<sub>📑 강의 슬라이드 p.24</sub>
</div>

## 6.2 표본평균 분포 시뮬레이션 (n = 100, 400, 625)

표본 크기가 커질수록 표본평균의 분포가 점점 좁아지는 것을 확인한다.

<div align="center">
<img src="images/page_25.png" width="760" alt="강의 슬라이드 p.25"><br>
<sub>📑 강의 슬라이드 p.25</sub>
</div>

In [ ]:
pop_mean = np.mean(delay)
pop_sd = np.std(delay)
print('모집단 평균:', round(pop_mean, 4))
print('모집단 SD  :', round(pop_sd, 4))

def simulate_sample_mean(data, sample_size, repetitions=10000):
    means = np.array([np.mean(np.random.choice(data, sample_size))
                      for _ in range(repetitions)])
    return means

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, n in zip(axes, [100, 400, 625]):
    sm = simulate_sample_mean(delay, n)
    hist_percent(sm, bins=np.arange(5, 35, 1), xlabel='Sample Mean', ax=ax)
    ax.set_title(f'Sample Size {n}')
    print(f'n={n:4d}  표본평균들의 SD = {np.std(sm):.4f}',
          f'  (예측: pop_SD/√n = {pop_sd/np.sqrt(n):.4f})')
plt.tight_layout(); plt.show()

## 6.3 모든 표본 평균의 표준편차(SD) — 제곱근 법칙

$$\boxed{\;\text{SD(표본평균)} = \frac{\text{모집단 SD}}{\sqrt{\text{표본크기}}}\;}$$

비행 지연 데이터(모집단 SD ≈ 39.48분) 로 검증:
- n=100 → 39.48/√100 = 3.95
- n=400 → 39.48/√400 = 1.97
- n=625 → 39.48/√625 = 1.58

시뮬레이션 값과 $\text{pop\_sd}/\sqrt{n}$ 그래프가 거의 완벽히 일치 → 공식 확인

<div align="center">
<img src="images/page_26.png" width="760" alt="강의 슬라이드 p.26"><br>
<sub>📑 강의 슬라이드 p.26</sub>
</div>

In [ ]:
sample_sizes = np.arange(25, 626, 25)
emp_sd = np.array([np.std(simulate_sample_mean(delay, n, repetitions=2000))
                   for n in sample_sizes])
predicted = pop_sd / np.sqrt(sample_sizes)

sd_comparison = pd.DataFrame({'Sample Size n': sample_sizes,
                              'SD of Sample Means (sim)': np.round(emp_sd, 4),
                              'pop_sd / sqrt(n)': np.round(predicted, 4)})
print(sd_comparison.head(10).to_string(index=False))

plt.plot(sample_sizes, predicted, label='pop_sd / √n (공식)', color='red')
plt.scatter(sample_sizes, emp_sd, s=15, label='시뮬레이션', color='#3b5d78')
plt.xlabel('Sample Size n'); plt.ylabel('SD of Sample Means')
plt.title('제곱근 법칙 검증'); plt.legend()
plt.show()

## 6.4 최종 통합 정리

**정리**
- 모집단 크기 무관: 공식에 모집단 크기 N 이 없다 → 같은 n 이면 같은 정확도
- 표본 크기 ↑ → 정확도 ↑: 분모의 √n 때문에 n 이 커질수록 SD(평균)가 감소
- 모집단 SD 는 상수: n 을 바꿔도 변하지 않음

**제곱근 법칙**: 표본 크기를 k배 늘리면 SD(표본평균)은 $1/\sqrt{k}$ 배로 줄어듦 → 정확도는 √k배 향상

**대형 랜덤 표본(복원 추출)에서 표본평균의 확률 분포**
- 근사 정규분포
- 중심 = 모집단 평균
- 표준편차 = 모집단 SD / √n
- 위 3가지 모두 모집단 분포의 형태와 무관하게 성립

<div align="center">
<img src="images/page_27.png" width="760" alt="강의 슬라이드 p.27"><br>
<sub>📑 강의 슬라이드 p.27</sub>
</div>

---
# 7. 표본크기 결정

## 7.1 개요

- 실제 조사에서는 표본을 수집하기 **전에** 크기를 결정해야 한다 (수집 후 "부족했다"는 것을 알면 늦는다).
- 표본이 클수록 정확도는 높지만 비용·시간이 증가 → 원하는 정확도를 달성하는 **최솟값**을 찾는 것이 목표
- 공식 $\text{SD}(\bar{x}) = \text{모집단 SD} / \sqrt{n}$ 을 **역으로** 활용
- "n 이 주어졌을 때 정확도?" → "원하는 정확도를 위해 n 이 얼마여야?"

<div align="center">
<img src="images/page_29.png" width="760" alt="강의 슬라이드 p.29"><br>
<sub>📑 강의 슬라이드 p.29</sub>
</div>

## 7.2 예시 — 여론조사 시나리오

후보 A 지지율 추정을 위한 여론조사. 다음 조건에서 필요한 최소 표본 크기를 구한다.
- (가정) 유권자 수가 매우 많아 복원·비복원 추출 결과가 사실상 동일
- (추정방법) CLT 근사 정규분포를 활용한 95% 신뢰구간 사용
- (정확도 목표) 95% CI 의 전폭 ≤ 2% (반폭 ≤ 1%)

**5단계 풀이 (95% CI 폭에서 역산)**
1. 95% CI 폭 = 4 × SD(표본비율)  (정규곡선 "평균 ± 2 SD" = 95% 포함)
2. SD(표본비율) = SD(0–1 모집단) / √n
3. 부등식: 4 × (모집단 SD / √n) ≤ 0.01
4. √n ≥ 4 × 모집단 SD / 0.01 — 하지만 모집단 SD 를 모름
5. 0–1 모집단의 SD 상한(≤ 0.5) 활용 → 보수적 최소 표본 크기 계산

→ 지지율 p 를 모르기 때문에 SD 도 모름. 해결책은 **어떤 p 에 대해서도 성립하는 SD 상한**을 이용하는 것이다.

<div align="center">
<img src="images/page_30.png" width="760" alt="강의 슬라이드 p.30"><br>
<sub>📑 강의 슬라이드 p.30</sub>
</div>

## 7.3 0–1 모집단의 SD — 왜 최댓값이 0.5인가?

비율 p 인 0–1 모집단(p 만큼 1, (1−p) 만큼 0):
- 평균 $= 1\cdot p + 0\cdot(1-p) = p$
- 분산 $= (1-p)^2 p + (0-p)^2 (1-p) = p(1-p)$
- $\text{SD} = \sqrt{p(1-p)}$

$\frac{d}{dp}[p - p^2] = 1 - 2p = 0 \Rightarrow p = 0.5$ 에서 최대.
최댓값 $\text{SD} = \sqrt{0.5 \times 0.5} = 0.5$

<div align="center">
<img src="images/page_31.png" width="760" alt="강의 슬라이드 p.31"><br>
<sub>📑 강의 슬라이드 p.31</sub>
</div>

In [ ]:
ps = np.arange(0.1, 0.91, 0.01)
sd_01 = np.sqrt(ps * (1 - ps))

table = pd.DataFrame({'p': [0.1, 0.3, 0.5, 0.7, 0.9]})
table['1-p'] = 1 - table['p']
table['p(1-p)'] = table['p'] * table['1-p']
table['SD'] = np.sqrt(table['p(1-p)'])
print(table.to_string(index=False))

plt.plot(ps, sd_01, color='#3b5d78')
plt.scatter([0.5], [0.5], color='red', zorder=5, label='최댓값 p=0.5, SD=0.5')
plt.xlabel('Population Proportion of 1\'s (p)'); plt.ylabel('Population SD')
plt.title('0–1 모집단의 SD = √(p(1-p))'); plt.legend()
plt.show()

## 7.4 두 경우 비교 — 왜 p=0.5 에서 가장 퍼지는가?

| | Case 1 (0과 1이 반반) | Case 2 (90% 1) |
|---|---|---|
| 데이터 | `[1 1 1 1 0 0 0 0]` | `[1 1 1 1 1 1 1 0]` |
| 평균 | 0.5 | 0.9 |
| 편차 | 평균보다 ±0.5 만큼 떨어짐 | 대부분 평균 가까이 몰림 |
| 표준편차 | SD = 0.5 (최대) | SD ≈ 0.3 |

반반 섞여 있으면 → 가장 퍼져 있음 → SD 최대.

<div align="center">
<img src="images/page_32.png" width="760" alt="강의 슬라이드 p.32"><br>
<sub>📑 강의 슬라이드 p.32</sub>
</div>

In [ ]:
case1 = np.array([1, 1, 1, 1, 0, 0, 0, 0])  # 반반
case2 = np.array([1, 1, 1, 1, 1, 1, 1, 0])  # 90% 1
for name, c in [('Case 1 (반반)', case1), ('Case 2 (90% 1)', case2)]:
    print(f'{name:16s} 평균={np.mean(c):.2f}  SD={np.std(c):.4f}')

## 7.5 최소 표본 크기 도출

1. 조건: CI 전폭 ≤ 0.02 → $4 \times \text{pop\_SD} / \sqrt{n} \le 0.02$
2. pop_SD 상한 0.5 대입: $4 \times 0.5 / \sqrt{n} \le 0.02$
3. 정리: $2 / \sqrt{n} \le 0.02 \Rightarrow \sqrt{n} \ge 100$
4. 양변 제곱: $n \ge 100^2 = 10000$

→ **CI 전폭 2% 이하의 95% CI 를 보장하려면 최소 10,000명 표본 필요.**

→ p 사전 정보가 있으면 더 작은 n 으로도 충분 — 하지만 틀렸을 경우 목표 정확도를 보장할 수 없다. 안전을 원한다면 항상 0.5 상한을 사용.

<div align="center">
<img src="images/page_33.png" width="760" alt="강의 슬라이드 p.33"><br>
<sub>📑 강의 슬라이드 p.33</sub>
</div>

In [ ]:
def min_sample_size(pop_sd, ci_total_width=0.02):
    # 4 * pop_sd / sqrt(n) <= ci_total_width
    sqrt_n = 4 * pop_sd / ci_total_width
    return int(np.ceil(sqrt_n ** 2))

rows = []
for desc, p_sd in [('p 완전히 모름 (보수적)', 0.5),
                   ('p ≈ 0.3 or 0.7 예상', np.sqrt(0.3*0.7)),
                   ('p ≈ 0.1 or 0.9 예상', np.sqrt(0.1*0.9)),
                   ('p = 0.5 로 예상', 0.5)]:
    n = min_sample_size(p_sd)
    rows.append([desc, round(p_sd, 3), n])

print(pd.DataFrame(rows, columns=['전제 조건', '사용 pop_SD', '필요 n (전폭 2%)']).to_string(index=False))
print()
print('보수적(p 모름) 최소 표본 크기:', min_sample_size(0.5), '명')

---
# 정리 (Q & A)

- **평균**은 분포의 무게중심(지렛점)이며, 우편향이면 평균 > 중앙값.
- **SD** 는 데이터가 평균에서 얼마나 퍼졌는지의 척도. 체비쇼프 부등식은 어떤 분포에서도 성립하는 하한을 준다.
- 종 모양 분포에서는 **표준 정규곡선**과 68–95–99.7 법칙으로 정확한 비율을 알 수 있다.
- **중심극한정리**: 모집단 분포와 무관하게 대형 표본의 합/평균은 근사 정규분포.
- **표본평균의 SD = 모집단 SD / √n** (제곱근 법칙).
- **표본크기 결정**: 0–1 모집단 SD 상한 0.5 를 쓰면, 95% CI 전폭 2% 보장에 최소 **10,000명** 필요.

### 연습 문제
1. NBA `Weight` 컬럼의 평균·SD 를 구하고, 평균 ± 2 SD 안에 실제로 몇 %가 들어오는지 체비쇼프 하한과 비교해 보세요.
2. United 비행 지연에서 표본 크기 n=100 과 n=2500 의 표본평균 SD 를 시뮬레이션하고, 비율이 제곱근 법칙(1/5배)과 맞는지 확인하세요.
3. 95% CI 전폭을 1% (반폭 0.5%) 로 줄이려면 최소 표본 크기는 몇 명일까요? `min_sample_size` 를 이용해 계산하세요.